In [ ]:
# Importer les packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
# Supprimer les warning
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### <font color = 'teal'>Importer et Lire les données<font>

In [ ]:
# importer les données
df = pd.read_csv('/content/drive/MyDrive/Telephone_data.csv')
df.head()

In [ ]:
# dimension des données
df.shape

In [ ]:
# infos des données
df.info()

### <font color = 'teal'>Exploration et preparation des données<font>



###### <font color = 'bisque'> Selection des variables catégorielles et quantitatives

In [ ]:
# selection les variables catégorielles
cat_data = df.select_dtypes(exclude = 'number')
cat_data.head()

In [ ]:
# selection des variables quantitatives
num_data = df.select_dtypes(exclude = 'object')
num_data.head()

###### <font color = 'bisque'> Exploration des variables catégorielles

In [ ]:
# décrire les variables catégorielles
cat_data.describe().T

In [ ]:
# fonction de countplot
import plotly.express as px
def countplot_func(col):
  val = df[col].value_counts()
  fig = px.bar(x=val.index, y=val.values, color = val.index.astype(str), text_auto=True)
  fig.show()

In [ ]:
# Diagrammes en barre des variables catégorielles
for col in cat_data :
  countplot_func(col)

###### <font color = 'bisque'> Exploration des variables quantitatives

In [ ]:
# décrire les variables quantitatives
num_data.describe().T

In [ ]:
# pairplot des variables quantitatives
sns.pairplot(num_data)
plt.show()

In [ ]:
#matrice de correlation
sns.heatmap(num_data.corr(), annot=True)
plt.show()

In [ ]:
# fonction de visualisation des boxplots
def boxplot_func(col):
  plt.figure(figsize=(25,3))
  sns.boxplot(x=df[col])
  plt.show()

In [ ]:
# boxplots des variables quantitatives
for col in num_data.columns:
  boxplot_func(col)

In [ ]:
# Enregistrer les valeurs uniques des variables catégorielles dans un objet joblib
import joblib as jb
uniques = []
for col in cat_data.columns:
  uniques.append(np.unique(df[col]).tolist())
jb.dump(uniques, 'uniques.joblib')

In [ ]:
# Encoder les variables categorielles
from sklearn.preprocessing import LabelEncoder

# Créer un encodeur par variable catégorielle
encoders = []
for col in cat_data.columns:
  encoder = LabelEncoder()
  df[col] = encoder.fit_transform(df[col])
  encoders.append(encoder)

# sauvegarde des encodeurs
jb.dump(encoders, 'encoders.joblib')

df.head()

###### <font color = 'bisque'> Preparation des données

In [ ]:
# Recherche du nombre optimal de voisins du KNNImputer avec GridSearchCV
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, KFold

# données dont les valeurs aberrantes (méthode IQR) sont remplacées par des NaN
df_nan = df.copy()
for col in num_data.columns:
  Q1 = df_nan[col].quantile(0.25)
  Q3 = df_nan[col].quantile(0.75)
  IQR = Q3 - Q1
  df_nan.loc[(df_nan[col] < Q1 - 1.5 * IQR) | (df_nan[col] > Q3 + 1.5 * IQR), col] = np.nan
x_nan = df_nan.drop('etat', axis=1)
y_nan = df_nan['etat']

# pipeline : KNNImputer puis modèle
pipe_knn = Pipeline([('imputer', KNNImputer()), ('model', RandomForestClassifier(n_estimators=50, random_state=42))])
# dictionnaire params : nombre de voisins à tester
params_knn = {'imputer__n_neighbors': [1, 3, 5, 7, 9, 11, 15]}
# validation croisée
cv_knn = KFold(n_splits=5, shuffle=True, random_state=42)
# instantier le modèle gridsearch
grid_knn = GridSearchCV(pipe_knn, params_knn, cv=cv_knn)
# lancer la recherche
grid_knn.fit(x_nan, y_nan)
# nombre optimal de voisins
best_k = grid_knn.best_params_['imputer__n_neighbors']
print('Nombre optimal de voisins :', best_k)

In [ ]:
from sklearn.impute import KNNImputer
# fonction d'imputation des valeurs aberrantes
def impute_func(df, col, n_neighbors=5):
    # 2. Détection des outliers (Méthode IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1 # interquantile
    min_value = Q1 - 1.5 * IQR
    max_value = Q3 + 1.5 * IQR

    # Les valeurs hors bornes deviennent des valeurs manquantes pour le KNNImputer
    df.loc[(df[col] < min_value) | (df[col] > max_value), col] = np.nan
    # n_neighbors : nombre de voisins pour calculer la moyenne de remplacement
    imputer = KNNImputer(n_neighbors=n_neighbors)
    df[:] = imputer.fit_transform(df)

In [ ]:
# imputation des valeurs extremes
for col in num_data:
  impute_func(df, col, n_neighbors=best_k)

In [ ]:
# Reafficher les boxplots
for col in df.columns:
  boxplot_func(col)

In [ ]:
# Fractionner les données en variables prédicteurs (x) et variable cibel(y)
x = df.drop('etat', axis=1).values
y = df.etat.values

In [ ]:
# normaliser les données
from sklearn.preprocessing import MinMaxScaler # Z = (X - Xmin) / (Xmax - Xmin)
scaler = MinMaxScaler()
x = scaler.fit_transform(x)
# sauvegarde du scaler
jb.dump(scaler, 'scaler.joblib')

x[:5]

In [ ]:
# splitter les données en train, val et test
# train (80%), val(10%) et test(10%)
#train : données d'entrainement, val: données utiliser pour évaluer chaque et test: données pour tester le modèle le plus performant
from sklearn.model_selection import train_test_split
x_train, x_val_test, y_train, y_val_test = train_test_split(x, y, train_size=0.8, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_val_test, y_val_test, train_size=0.5, random_state=42)

### <font color = 'teal'>Logistic Regression<font>

###### <font color = 'bisque'> Recherche des paramètre optimaux

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
# Recherche des paramètres optimaux pour la logistic regression
from sklearn.linear_model import LogisticRegression

#Validation croisée figé
cv = KFold(n_splits=5, shuffle=True, random_state=42)
# dictionnaire params
params={"C":np.logspace(-3,3,10),"penalty": ['l1','l2']}
# instantier le modèle Logistic regression
lg=LogisticRegression()
# instantier le modèle gridsearch
grid=GridSearchCV(lg,params,cv=cv)
# lancer la recherche
grid.fit(x_train,y_train)

In [ ]:
# Afficher les paramètres optimaux
grid.best_estimator_

###### <font color = 'bisque'> Entrainement du modèle

In [ ]:
# Entrainer la logistic regression en utilisant les paramètres optimaux
lg_model = LogisticRegression( C = 0.46415888336127775, penalty = 'l2')
lg_model.fit(x_train, y_train)

###### <font color ='bisque'>Evaluation du modèle

In [ ]:
# fonction d'évaluation
from sklearn.metrics import f1_score, accuracy_score
def evaluation(model):
  y_pred = model.predict(x_val)
  acc = accuracy_score(y_val, y_pred)
  f1 = f1_score(y_val, y_pred, average='micro')
  return acc, f1

In [ ]:
# Accuracy et F1 score du modèle sur les données de validation
acc_lg, f1_lg = evaluation(lg_model)
print(f"Accuracy : {acc_lg}, F1 score : {f1_lg}")

In [ ]:
# Importer ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# y_pred:  prédictions du modèle sur les données validation
def CM_visualisation(model_objet):

    y_pred = model_objet.predict(x_val)
    # Confusion matrix
    cnf_matr = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])

    # Utiliser ConfusionMatrixDisplay pour visualiser la matrice de confusion
    disp = ConfusionMatrixDisplay(confusion_matrix=cnf_matr,
                                  display_labels=['Occasion','Neuf', 'Reconditionné','Venant'])
    disp.plot() # Afficher la matrice de confusion
    plt.title('Matrice de confusion') # Définir le titre
    plt.show() # Afficher la figure

In [ ]:
# Matrice de confusion
CM_visualisation(lg_model)

### <font color = 'teal'>Decision Tree<font>

###### <font color = 'bisque'> Recherche des paramètre optimaux

In [ ]:
# Recherche des paramètres optimaux pour le Decision Tree
from sklearn.tree import DecisionTreeClassifier
# dictionnaire des params
param__ = {'criterion':['gini','entropy'],'max_depth':np.arange(3,15)}
# instantier le modèle Decision Tree
dt = DecisionTreeClassifier()
# instantier le modèle gridsearch
grid__ = GridSearchCV(dt, param__, cv = cv)
# entrainer le modèle
grid__.fit(x_train, y_train)

In [ ]:
# afficher les paramètres optimaux
grid__.best_params_

###### <font color = 'bisque'> Entrainement du modèle

In [ ]:
# Entrainer le modèle en utilisant les paramètres optimaux
dt_model = DecisionTreeClassifier(criterion = 'gini', max_depth = 13)
dt_model.fit(x_train, y_train)

###### <font color ='bisque'>Evaluation du modèle<font>

In [ ]:
# Accuracy et F1 score du modèle sur les données de validation
acc_dt, f1_dt = evaluation(dt_model)
print(f"Accuracy : {acc_dt}, F1 score : {f1_dt}")

In [ ]:
# Matrice de confusion
CM_visualisation(dt_model)

### <font color = 'teal'>Random Forest<font>

In [ ]:
# Recherche des paramètres optimaux pour le Random Forest
from sklearn.ensemble import RandomForestClassifier
# dictionnaire des  params
params = param_grid = {
    'n_estimators': list(np.arange(10,100,10)),
    'max_depth' : list(np.arange(3,15)),
    'criterion' :['gini', 'entropy']
}
# instantier le modèle Random Forest
rf = RandomForestClassifier()
# instantier le modèle gridsearch
grid = GridSearchCV(estimator=rf, param_grid=params, cv = cv)
# entrainer le modèle
grid.fit(x_train, y_train)

In [ ]:
# Afficher les paramètres optimaux
grid.best_params_

In [ ]:
# Entrainer Random Forest en utilisant les paramètres optimaux
rf_model = RandomForestClassifier(criterion = 'entropy', max_depth = 13, n_estimators = 30)
rf_model.fit(x_train, y_train)

In [ ]:
# télécharger le modèle le plus performant
jb.dump(rf_model, 'rf_model.joblib')

##### <font color ='bisque'>Evaluation du Random Forest<font>

In [ ]:
# Accuracy et F1 score du modèle sur les données de validation
acc_rf, f1_rf = evaluation(rf_model)
print(f"Accuracy : {acc_rf}, F1 score : {f1_rf}")

In [ ]:
# Matrice de confusion
CM_visualisation(rf_model)

In [ ]:
#diagrammes des performances
list_name = ['LG', 'RF', 'DT']
list_acc = [acc_lg, acc_rf, acc_dt]
list_f1 = [f1_lg, f1_rf, f1_dt]
list_final = [list_acc, list_f1]

for i in range(2):
  fig = px.bar(y=list_name, x=list_final[i], color=list_name)
  fig.show()

### <font color ='bisque'>Tester le modèle le plus performant<font>

In [ ]:
# echantillons
x_test10 = x_test[:10]
y_test10 = y_test[:10]

#predictions
y_pred10 = rf_model.predict(x_test10)
classnames = ['occasion', 'neuf', 'reconditionné', 'venant']

for i in range(10):
  print(f'Prédiction {str(i + 1)} : {classnames[int(y_pred10[i])]} --- {classnames[int(y_test10[i])]}')

### <font color = 'teal'>Déploiement avec Gradio<font>

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import joblib as jb
import pandas as pd
import numpy as np
# importer les encodeurs
encoders = jb.load('encoders.joblib')
# importer le best modèle
rf_model = jb.load('rf_model.joblib')
# importer les valeurs uniques
uniques = jb.load('uniques.joblib')
# importer le scaler
scaler = jb.load('scaler.joblib')

def Pred_func(prix, adresse, marque, dim_ecr, ram, stockage):

  # Encoder les valeurs de adresse et marque
  adresse = encoders[0].transform([adresse])[0]
  marque = encoders[1].transform([marque])[0]
  # vecteur des valeurs numériques
  x_new = np.array([prix, adresse, marque, dim_ecr, ram, stockage])
  x_new = x_new.reshape(1,-1) # convert en un 2D array
  #normaliser
  x_new = scaler.transform(x_new)
  # Prédire
  y_pred = rf_model.predict(x_new)
  # Retrouver le nom de la classe
  return encoders[2].inverse_transform([int(y_pred[0])])[0]


# Fonction de prédiction multiple
def Pred_func_csv(file):
  # Lire le fichier csv
  df = pd.read_csv(file)
  predictions = []
  # Boucle sur les lignes du dataframe
  for row in df.iloc[:, :].values:
    y_pred = Pred_func(row[0], row[1], row[2], row[3], row[4], row[5])
    predictions.append(y_pred)

  df['etat_prediction'] = predictions
  df.to_csv('predictions.csv', index = False)
  return 'predictions.csv'

# définir les blocks
demo = gr.Blocks(theme='shivi/calm_seafoam')

# Créer les inputs
inputs = [gr.Number(label='Prix'),
          gr.Dropdown(choices=uniques[0], label='Adresse'),
          gr.Dropdown(choices=uniques[1], label='Marque'),
          gr.Number(label='Dimension écran'),
          gr.Number(label='RAM'),
          gr.Number(label='Stockage')]
# Créer les outputs
outputs = gr.Textbox(label='Etat du téléphone')
# Créer l'interface 1
interface1 = gr.Interface(fn = Pred_func,
                         inputs = inputs,
                         outputs = outputs,
                         title="Prédire l'état d'un téléphone avec une seule entrée",
                         description = """Ce modèle de machine learning permet de prédire l'état d'un téléphone
                         (Occasion, Neuf, Reconditionné, Venant) à partir du prix, de l'adresse, de la marque,
                         de la dimension de l'écran, de la RAM et du stockage.
                         """)
# Créer l'interface 2
interface2 = gr.Interface(fn = Pred_func_csv,
                         inputs = gr.File(label='Upload a csv file'),
                          outputs = gr.File(label='Download a csv file'),
                         title="Prédire l'état de téléphones avec plusieurs entrées",
                         description = """Ce modèle de machine learning permet de prédire l'état de téléphones
                         à partir d'un fichier csv (prix, adresse, marque, dim_ecr, ram, stockage).
                         """)

# faire un tabbing des interfaces
with demo:
  gr.TabbedInterface([interface1, interface2], ['Simple Prédiction', 'Prédiction multiple'])

# lancer l'interface
demo.launch()